## Import packages

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Define data paths

In [ ]:
data_dir = '../data/chest_xray'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# Verify directories exist
for d in [train_dir, val_dir, test_dir]:
    print(f"{d}: {'EXISTS' if os.path.exists(d) else 'NOT FOUND'}")

## Count images per class

In [ ]:
def count_images(directory):
    """Count images in each class subdirectory."""
    counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len([f for f in os.listdir(class_path)
                        if f.lower().endswith(('.jpeg', '.jpg', '.png'))])
            counts[class_name] = count
    return counts

train_counts = count_images(train_dir)
val_counts = count_images(val_dir)
test_counts = count_images(test_dir)

print("Dataset Distribution:")
print(f"{'Split':<10} {'NORMAL':<10} {'PNEUMONIA':<12} {'Total':<10}")
print("-" * 42)
print(f"{'Train':<10} {train_counts.get('NORMAL', 0):<10} {train_counts.get('PNEUMONIA', 0):<12} {sum(train_counts.values()):<10}")
print(f"{'Val':<10} {val_counts.get('NORMAL', 0):<10} {val_counts.get('PNEUMONIA', 0):<12} {sum(val_counts.values()):<10}")
print(f"{'Test':<10} {test_counts.get('NORMAL', 0):<10} {test_counts.get('PNEUMONIA', 0):<12} {sum(test_counts.values()):<10}")
print("-" * 42)
total = sum(train_counts.values()) + sum(val_counts.values()) + sum(test_counts.values())
print(f"{'Total':<10} {'':<10} {'':<12} {total:<10}")

## Plot class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (split, counts) in zip(axes, [('Train', train_counts), ('Val', val_counts), ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(), color=['steelblue', 'coral'])
    ax.set_title(f'{split} Set Distribution')
    ax.set_ylabel('Number of Images')
    for bar, count in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(count), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Show imbalance ratio
if 'PNEUMONIA' in train_counts and 'NORMAL' in train_counts:
    ratio = train_counts['PNEUMONIA'] / train_counts['NORMAL']
    print(f"\nClass imbalance ratio (Pneumonia:Normal): {ratio:.2f}:1")

## Display sample Normal images

In [ ]:
normal_dir = os.path.join(train_dir, 'NORMAL')
normal_images = [f for f in os.listdir(normal_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample NORMAL Chest X-Rays', fontsize=14)

for ax, img_name in zip(axes.flatten(), normal_images):
    img = Image.open(os.path.join(normal_dir, img_name))
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Display sample Pneumonia images

In [ ]:
pneumonia_dir = os.path.join(train_dir, 'PNEUMONIA')
pneumonia_images = [f for f in os.listdir(pneumonia_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample PNEUMONIA Chest X-Rays', fontsize=14)

for ax, img_name in zip(axes.flatten(), pneumonia_images):
    img = Image.open(os.path.join(pneumonia_dir, img_name))
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Side by side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

normal_img = Image.open(os.path.join(normal_dir, normal_images[0]))
pneumonia_img = Image.open(os.path.join(pneumonia_dir, pneumonia_images[0]))

axes[0].imshow(normal_img, cmap='gray')
axes[0].set_title('NORMAL', fontsize=14, color='green')
axes[0].axis('off')

axes[1].imshow(pneumonia_img, cmap='gray')
axes[1].set_title('PNEUMONIA', fontsize=14, color='red')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Analyze image sizes

In [ ]:
def get_image_sizes(directory, max_images=200):
    """Get dimensions of images in a directory."""
    widths, heights = [], []
    count = 0
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for img_name in os.listdir(class_path):
            if not img_name.lower().endswith(('.jpeg', '.jpg', '.png')):
                continue
            img = Image.open(os.path.join(class_path, img_name))
            widths.append(img.size[0])
            heights.append(img.size[1])
            count += 1
            if count >= max_images:
                return widths, heights
    return widths, heights

widths, heights = get_image_sizes(train_dir, max_images=500)

print(f"Image size statistics (sampled {len(widths)} images):")
print(f"Width  — Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
print(f"Height — Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")

## Plot image size distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Count')

axes[1].hist(heights, bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Pixel intensity analysis

In [ ]:
def get_pixel_stats(directory, class_name, n_images=50):
    """Get mean pixel intensity for a class."""
    intensities = []
    class_path = os.path.join(directory, class_name)
    images = os.listdir(class_path)[:n_images]
    for img_name in images:
        if not img_name.lower().endswith(('.jpeg', '.jpg', '.png')):
            continue
        img = Image.open(os.path.join(class_path, img_name)).convert('L')
        img_array = np.array(img)
        intensities.append(img_array.mean())
    return intensities

normal_intensities = get_pixel_stats(train_dir, 'NORMAL')
pneumonia_intensities = get_pixel_stats(train_dir, 'PNEUMONIA')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_intensities, bins=20, alpha=0.6, label='Normal', color='steelblue')
ax.hist(pneumonia_intensities, bins=20, alpha=0.6, label='Pneumonia', color='coral')
ax.set_title('Mean Pixel Intensity Distribution by Class')
ax.set_xlabel('Mean Pixel Intensity')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Normal mean intensity: {np.mean(normal_intensities):.2f}")
print(f"Pneumonia mean intensity: {np.mean(pneumonia_intensities):.2f}")

## Summary

In [ ]:
print("=" * 50)
print("EXPLORATION SUMMARY")
print("=" * 50)
print(f"Total images: {total}")
print(f"Training: {sum(train_counts.values())}")
print(f"Validation: {sum(val_counts.values())} (needs expansion)")
print(f"Test: {sum(test_counts.values())}")
print(f"Classes: NORMAL, PNEUMONIA")
print(f"Imbalance ratio: {ratio:.2f}:1 (Pneumonia:Normal)")
print(f"Image sizes vary — will resize to 224x224")
print("=" * 50)
print("\nKey observations:")
print("- Dataset is imbalanced — need class weights")
print("- Validation set is too small — need to split from training")
print("- Images have varying sizes — need uniform resizing")
print("- Pneumonia X-rays tend to have more white/opaque areas")
print("\nNext: 02_preprocessing.ipynb")